In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns  # for a heatmap of correlations
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX

# ===== Step 1: Load & Preprocess the Data =====
# File path to the Excel workbook
file_path = 'Sea_Ice_Index_Monthly_Data_with_Statistics_G02135_v3.0.xlsx'
xls = pd.ExcelFile(file_path)
dfs = []
2
for sheet in xls.sheet_names:
    if sheet != 'Documentation':
        # Read the specified columns from each sheet (starting at row 10)
        df = pd.read_excel(file_path, sheet_name=sheet, header=9, 
                           usecols=["year", "month", "extent", "area", "extent-anomaly", 
                                    "trend-through-year-km^2-per-year"])
        dfs.append(df)
        
# Combine all sheets into one DataFrame
combined_df = pd.concat(dfs, ignore_index=True)

# ===== Step 2: Load & combine CO₂ + CH₄ =====
ghg_path = 'CO2_CH4_conc(1983).xlsx'

co2 = pd.read_excel(ghg_path, sheet_name='CO2')
ch4 = pd.read_excel(ghg_path, sheet_name='CH4')

# Rename and keep only year/month + concentration
co2 = co2.rename(columns={'Year':'year','Month':'month','average NH':'avg_CO2_ppm'})
ch4 = ch4.rename(columns={'Year':'year','Month':'month','average NH':'avg_CH4_ppb'})

# Merge CO₂ + CH₄ into a single table
ghg = pd.merge(co2[['year','month','avg_CO2_ppm']],
               ch4[['year','month','avg_CH4_ppb']],
               on=['year','month'], how='inner')

# ===== Step 3: Trim Sea Ice to start from July 1983 =====
mask = (combined_df['year'] >= 1983)  & (combined_df['month'] >= 7)
sea_trimmed = combined_df[mask].reset_index(drop=True)

# ===== Step 4: Merge Sea Ice + GHG =====
merged = pd.merge(sea_trimmed, ghg, on=['year','month'], how='inner')


merged.to_excel('SeaIce_CO2_CH4_Merged.xlsx', index=False)

# ===== Step 5: Create a Date Index =====
# Convert year and month into a datetime index (day set to 1)
merged['date'] = pd.to_datetime(dict(year=merged['year'].astype(int), 
                                          month=merged['month'].astype(int), 
                                          day=1))
merged.set_index('date', inplace=True)
merged.sort_index(inplace=True)
merged.to_excel('Transformed Data.xlsx', index=True)


In [ ]:
from pathlib import Path
from IPython.display import Image, display

from src.arctic_sea_ice_analysis import main

# Regenerate corrected datasets, JPEG figures, validation metrics, and forecast output.
main()

figure_order = [
    'extent_time_series.jpg',
    'avg_CO2_ppm_time_series.jpg',
    'avg_CH4_ppb_time_series.jpg',
    'correlation_matrix.jpg',
    'monthly_extent_distribution.jpg',
    'melt_season_extent.jpg',
    'seasonal_decomposition.jpg',
    'sarimax_12_month_forecast.jpg',
]

for figure_name in figure_order:
    figure_path = Path('outputs/figures') / figure_name
    if figure_path.exists():
        print(f'
{figure_name}')
        display(Image(filename=str(figure_path)))
